<a href="https://colab.research.google.com/github/tommbk50-hub/Cheminformatics/blob/main/pyOpenMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Why use OpenMS

OpenMS is an open-source, C++ framework hosted on Github <https://github.com/OpenMS/OpenMS> for analyzing large volumes of mass spectrometry data. It has been specially designed for analyzing high performance LC-MS data but over recent times, has been extended to analyze data generated by other techniques.

https://pyopenms.readthedocs.io/en/latest/user_guide/background.html


Installing these programmes by uv is much faster than installing via pip. Here is why uv excels with this specific stack:

1. **Massive Parallel Downloads & Compilations:** Libraries like rdkit, matchms, ms2deepscore, and their dependencies (numba, scipy, matplotlib, torch) are heavy. Standard pip downloads and builds them sequentially, whereas uv processes downloads, metadata extraction, and wheel building in parallel.

2. **Rust-Powered Engine:** uv is written in Rust and operates directly at the system level. It bypasses Python's interpreter overhead for resolving dependency graphs.

3. **Immediate Mismatch Resolution:** In your notebook's install log, pip spent significant time downloading and building packages before finally failing late with: ERROR: Package 'neims-pytorch' requires a different Python: 3.13.16 not in '==3.12.*'. uv resolves the entire dependency tree before downloading any binaries. It would have detected this environment conflict within milliseconds, saving you minutes of wasted download time.

In [ ]:
#@title 1. Install required Python libraries with pinned versions
!uv pip install --system -q \
    "ipyketcher" \
    "rdkit>=2023.9.1" \
    "mols2grid==2.0.0" \
    "pyteomics==4.6.3" \
    "pyopenms>=3.1.0" \
    "psims==1.3.1" \
    "lxml>=5.1.0" \
    "hdf5plugin==4.3.0" \
    "pandas>=2.0.0,<3.0.0" \
    "pubchempy==1.0.4" \
    "mendeleev" \
    "marimo==0.2.5" \
    "wigglystuff==0.1.3" \
    "matchms" \
    "onnxruntime>=1.17.1" \
    "bottleneck==1.3.8" \
    "plotly>=5.18.0"

!pip install kaleido==0.1.0

import os
import glob
import pandas as pd
import matplotlib.colors as mcolors

# Helper to convert matplotlib colors to Plotly-compatible RGB strings
def to_plotly_color(color_val):
    """Converts any matplotlib-compatible color (including RGB tuples) to an rgb(r,g,b) string for Plotly."""
    rgba = mcolors.to_rgba(color_val)
    return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"

# 1. Clone hplc-py for chromatographic deconvolution
if not os.path.exists('hplc-py'):
    !git clone -q https://github.com/cremerlab/hplc-py.git

In [ ]:
#@title 2. Plot HPLC Chromatogram, Convert the HPLC Trace into a Pandas dataframe
import urllib.request
import os
import matplotlib.pyplot as plt
from pyteomics import mzml
import pandas as pd

# 1. Download the new dataset
url_new = "https://raw.githubusercontent.com/OpenMS/OpenMS/develop/src/tests/topp/FeatureFinderMetaboIdent_1_input.mzML"
filename_new = "FeatureFinderMetaboIdent_1_input.mzML"

if not os.path.exists(filename_new):
    print(f"Downloading {filename_new}...")
    urllib.request.urlretrieve(url_new, filename_new)

# 2. Parse the new dataset
print(f"Parsing {filename_new}...")
rt_new = []
tic_new = []

try:
    with mzml.read(filename_new) as reader:
        for spectrum in reader:
            if spectrum['ms level'] == 1:
                scan = spectrum.get('scanList', {}).get('scan', [{}])[0]
                rt = scan.get('scan start time', 0)
                tic = spectrum.get('total ion current', 0)
                rt_new.append(rt)
                tic_new.append(tic)

    # Create a pandas DataFrame from the parsed data
    hplc_df = pd.DataFrame({
        'Signal': tic_new,
        'Time': rt_new
    })

    print("HPLC Trace DataFrame successfully created:")
    print(hplc_df.head())

    # 3. Plot the new HPLC trace using the DataFrame
    plt.figure(figsize=(10, 6))
    plt.plot(hplc_df['Time'], hplc_df['Signal'], color='green', linewidth=1.5)
    plt.title('HPLC Trace: FeatureFinderMetaboIdent_1_input.mzML')
    plt.xlabel('Retention Time (seconds)')
    plt.ylabel('Total Ion Current (Signal)')
    plt.grid(True)
    plt.tight_layout()
    # Cache this Figure (before matplotlib's inline backend auto-closes it after
    # this cell finishes) so later cells can feed it into Gemini's vision context.
    globals().setdefault('_notebook_figure_registry', {})['raw_hplc_chromatogram'] = plt.gcf()

    plt.show()

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
#@title 3. List SMILES to find
import pandas as pd
import os
import sys
import traitlets
import subprocess
import ipywidgets as widgets
from rdkit import Chem
from rdkit.Chem import Descriptors

# Dynamic patch to solve 'TypeError: type "List" is not subscriptable' on Python 3.13 inside ipyketcher[cite: 1]
if not hasattr(traitlets.List, '__class_getitem__'):
    traitlets.List.__class_getitem__ = classmethod(lambda cls, item: cls)

# --- PATCH IPYKETCHER BUG ---[cite: 1]
ipk_init = f"/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages/ipyketcher/__init__.py"
if os.path.exists(ipk_init):
    with open(ipk_init, 'r') as f:
        content = f.read()
    if "traitlets.List[str]()" in content:
        with open(ipk_init, 'w') as f:
            f.write(content.replace("traitlets.List[str]()", "traitlets.List()"))
# ----------------------------

# Force reload in case the broken version was already cached in memory[cite: 1]
if 'ipyketcher' in sys.modules:
    del sys.modules['ipyketcher']
import ipyketcher
from IPython.display import display, HTML

# 1. Initialize and display the molecular sketching canvas[cite: 1]
editor = ipyketcher.KetcherEditor()
print("🎨 Canvas initialized. (Draw molecules to add them to your interactive table below)")
display(editor)

# Helper function to calculate exact mass via RDKit
def calculate_exact_mass(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol:
        return f"{Descriptors.ExactMolWt(mol):.4f}"
    return "Invalid"

# Define reaction scheme list dynamically with the requested molecules[cite: 1]
reaction_smiles = [
    ("adenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
    ("n-deoxyadenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)C1"),
    ("n-inosine", "Product", "O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
    ("n2-o-methylcytidine", "Product", "CO[C@@H]1[C@H](O[C@H](CO)[C@@H]1O)n2ccc(N)nc2=O"),
    ("n5-methylcytidine", "Product", "Cc1cn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)nc1N"),
    ("n5-formylcytidine", "Product", "Nc1nc(=O)n([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)cc1C=O")
]

# Create a UI Row for the interactive table
def create_table_row(comp_no, name, classification, smiles):
    w_no = widgets.HTML(value=f"<b>{comp_no}</b>", layout=widgets.Layout(width='40px', padding='0 5px'))
    w_name = widgets.Text(value=name, placeholder='Molecule Name', layout=widgets.Layout(width='160px', padding='0 5px'))
    w_class = widgets.Dropdown(
        options=['Reactant', 'Reagent', 'Product'],
        value=classification,
        layout=widgets.Layout(width='100px', padding='0 5px')
    )
    w_smiles = widgets.Text(value=smiles, layout=widgets.Layout(width='320px', padding='0 5px'))
    w_mass = widgets.HTML(value=calculate_exact_mass(smiles), layout=widgets.Layout(width='80px', padding='0 5px'))

    return widgets.HBox([w_no, w_name, w_class, w_smiles, w_mass])

# Build Table Header
header_row = widgets.HBox([
    widgets.HTML(value="<b>No.</b>", layout=widgets.Layout(width='40px', padding='0 5px')),
    widgets.HTML(value="<b>Name</b>", layout=widgets.Layout(width='160px', padding='0 5px')),
    widgets.HTML(value="<b>Classification</b>", layout=widgets.Layout(width='100px', padding='0 5px')),
    widgets.HTML(value="<b>SMILES String</b>", layout=widgets.Layout(width='320px', padding='0 5px')),
    widgets.HTML(value="<b>Exact Mass</b>", layout=widgets.Layout(width='80px', padding='0 5px'))
])

# Populate initial table rows
table_rows = [create_table_row(i + 1, name, cls, smi) for i, (name, cls, smi) in enumerate(reaction_smiles)]
table_container = widgets.VBox([header_row] + table_rows)

btn_update = widgets.Button(description='Update List', button_style='success')
out = widgets.Output()

def on_update_clicked(b):
    with out:
        out.clear_output()
        # ipyketcher stores current smiles in editor.smiles[cite: 1]
        drawn_smi = getattr(editor, 'smiles', None)
        if drawn_smi:
            # Check for duplicates based on existing SMILES in the table
            existing_smiles = [row.children[3].value for row in table_container.children[1:]]
            if drawn_smi not in existing_smiles:
                new_index = len(table_container.children) # Header counts as 1, so len equals the next compound number
                new_row = create_table_row(new_index, "Drawn Molecule", "Reactant", drawn_smi)

                # Append the new row to the table container
                table_container.children = list(table_container.children) + [new_row]
                print(f"Added to table: {drawn_smi}")
            else:
                print(f"Molecule already exists in the table: {drawn_smi}")
        else:
            print("No SMILES found in editor. Please draw a molecule first.")[cite: 1]

btn_update.on_click(on_update_clicked)

display(table_container)
display(widgets.HBox([btn_update]))
display(out)

Here we use an **LC-MS1 alignment/extraction pipeline**.

 We can use the OpenMS **Python wrapper, pyOpenMS**, directly inside this colab notebook to perform a precise, high-resolution Extracted Ion Chromatogram (EIC) extraction and retention time alignment for your target molecular masses.

In [ ]:
#@title 4. Unified Peak Detection, Dominant-m/z Confidence Scoring & Auto-Arranged Cards (dataframe = detected_df)

import subprocess
import sys
import os
import re
import warnings

try:
    import scipy.signal
except ImportError:
    print("🔧 Installing missing dependencies (scipy, numpy) via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "numpy", "scipy"], capture_output=True)
    import scipy.signal
    print("✅ Repair complete.")

try:
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
except ImportError:
    print("🔧 Installing missing dependencies (plotly, kaleido) via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "plotly", "kaleido"], capture_output=True)
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
    print("✅ Repair complete.")

# Dynamic explicit validation of kaleido dependency
try:
    import kaleido
except ImportError:
    print("🔧 Installing/reloading missing kaleido via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "kaleido"], capture_output=True)
    print("✅ Kaleido installation complete.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets
from pyteomics import mzml
import pyopenms as po
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Draw

warnings.filterwarnings('ignore')

# Plotly renders natively in Jupyter/VS Code notebooks; only Colab needs an explicit
# renderer override so the interactive, zoomable figures display inline.
if 'google.colab' in sys.modules:
    pio.renderers.default = 'colab'

# 1. Ensure hplc-py and pyOpenMS experiment are loaded
if 'hplc-py' not in sys.path:
    sys.path.append(os.path.abspath('hplc-py'))
from hplc.quant import Chromatogram

mzml_filename = "FeatureFinderMetaboIdent_1_input.mzML"
if not os.path.exists(mzml_filename):
    mzml_filename = "/content/FeatureFinderMetaboIdent_1_input.mzML"

if 'od' not in globals() and os.path.exists(mzml_filename):
    od = po.MSExperiment()
    po.FileHandler().loadExperiment(mzml_filename, od)

signal_raw = hplc_df['Signal'].values
signal_norm = (signal_raw - signal_raw.min()) / (signal_raw.max() - signal_raw.min())

# --- 2. Dynamic Hyperparameter Optimization (Elbow Method) ---
print("Running Dynamic Hyperparameter Optimization...")
prominence_range = np.logspace(-2, -0.5, 8)
buffer_val = 0
results = []
chrom_base = Chromatogram(hplc_df, cols={'time': 'Time', 'signal': 'Signal'}, time_window=[200, 300])

for p in prominence_range:
    try:
        peaks = chrom_base.fit_peaks(prominence=p, buffer=buffer_val)
        results.append((p, len(peaks)))
    except Exception:
        pass

if not results:
    default_prominence = 0.05
    print("Optimization failed. Using default prominence: 0.05")
else:
    proms_all = np.array([x[0] for x in results])
    counts_all = np.array([x[1] for x in results])

    max_idx = np.argmax(counts_all)
    valid_proms = proms_all[max_idx:]
    valid_counts = counts_all[max_idx:]

    if len(valid_proms) > 1:
        p1 = np.array([np.log10(valid_proms[0]), valid_counts[0]])
        p2 = np.array([np.log10(valid_proms[-1]), valid_counts[-1]])

        distances = []
        for i in range(len(valid_proms)):
            p3 = np.array([np.log10(valid_proms[i]), valid_counts[i]])
            if np.linalg.norm(p2 - p1) == 0:
                dist = 0
            else:
                cross_prod_2d = (p2[0] - p1[0]) * (p1[1] - p3[1]) - (p2[1] - p1[1]) * (p1[0] - p3[0])
                dist = np.abs(cross_prod_2d) / np.linalg.norm(p2 - p1)
            distances.append(dist)

        optimal_idx_relative = np.argmax(distances)
        optimal_prominence = valid_proms[optimal_idx_relative]
        optimal_count = valid_counts[optimal_idx_relative]
    else:
        optimal_prominence = valid_proms[0]
        optimal_count = valid_counts[0]

    default_prominence = optimal_prominence
    prominence_val = optimal_prominence

    display(Markdown("---"))
    display(Markdown(f"### ✅ Optimization Complete"))
    display(Markdown(f"**Optimal Prominence Threshold:** `{optimal_prominence:.4f}` | **Peaks Resolved:** `{optimal_count}`"))

    fig_opt, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [2.5, 1]})
    ax1.plot(proms_all, counts_all, marker='o', linestyle='-', color='#2c3e50', linewidth=2, label='Fitted Peaks')
    ax1.axvline(optimal_prominence, color='#e74c3c', linestyle='--', linewidth=2, label=f'Optimal Elbow ({optimal_prominence:.4f})')
    ax1.scatter([optimal_prominence], [optimal_count], color='#e74c3c', s=150, zorder=5)
    ax1.set_xscale('log')
    ax1.set_xticks(prominence_range)
    ax1.set_xticklabels([f"{x:.4f}" for x in prominence_range], rotation=45)
    ax1.set_title('Hyperparameter Optimization: Signal-to-Noise Ratio (L-Curve)', fontsize=13, pad=12)
    ax1.set_xlabel('Prominence Threshold (Log Scale)', fontsize=11)
    ax1.set_ylabel('Number of Deconvoluted Peaks', fontsize=11)
    ax1.legend(fontsize=10)
    ax1.grid(True, which="both", ls="--", alpha=0.4)

    ax2.axis('tight')
    ax2.axis('off')
    table_data = [[f"{p:.4f}", int(c)] for p, c in zip(proms_all, counts_all)]
    col_labels = ['Prominence Threshold', 'Number of\nDeconvoluted Peaks']
    table = ax2.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.1, 1.8)

    optimal_row_idx = list(proms_all).index(optimal_prominence) + 1
    for (row, col), cell in table.get_celld().items():
        if row == optimal_row_idx:
            cell.set_edgecolor('red')
            cell.set_linewidth(2)
    ax2.set_title('Tabular Data', fontsize=12, pad=12)
    plt.tight_layout()
    plt.show()

# --- 3. Candidate Fetching & Theoretical Ion / Isotope Generators ---
def get_candidates_from_table():
    if 'table_container' in globals():
        candidates = []
        for row in table_container.children[1:]:
            name = row.children[1].value.strip()
            classification = row.children[2].value
            smi = row.children[3].value.strip()
            if smi:
                candidates.append((name, classification, smi))
        if candidates:
            return candidates
    return [
        ("adenosine",           "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
        ("n-deoxyadenosine",    "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)C1"),
        ("n-inosine",           "Product", "O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
        ("n2-o-methylcytidine", "Product", "CO[C@@H]1[C@H](O[C@H](CO)[C@@H]1O)n2ccc(N)nc2=O"),
        ("n5-methylcytidine",   "Product", "Cc1cn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)nc1N"),
        ("n5-formylcytidine",   "Product", "Nc1nc(=O)n([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)cc1C=O")
    ]

esi_pos_adducts = [
    ("[M+H]+",       1, 1.007276,  1),
    ("[M+NH4]+",     1, 18.033823, 1),
    ("[M+Na]+",      1, 22.989218, 1),
    ("[M+CH3OH+H]+", 1, 33.033489, 1),
    ("[M+K]+",       1, 38.963158, 1),
    ("[M+ACN+H]+",   1, 42.033823, 1),
    ("[2M+H]+",      2, 1.007276,  1),
    ("[2M+Na]+",     2, 22.989218, 1),
]

insource_neutral_losses = [
    ("[M+H-Ribose]+",      -132.042259),
    ("[M+H-Deoxyribose]+", -116.047344),
    ("[M+H-MeRibose]+",    -146.057909),
    ("[M+H-H2O]+",         -18.010565),
]

def parse_dominant_mz_string(dom_str):
    if not dom_str or dom_str == "None":
        return []
    parsed = []
    for item in str(dom_str).split(","):
        m = re.search(r"([0-9]+\.[0-9]+)\s*\(([0-9.]+)%\)", item.strip())
        if m:
            parsed.append((float(m.group(1)), float(m.group(2))))
    return parsed

def compute_confidence_from_dominant_mz(best_group, dom_mz_list, full_window_spectrum):
    adducts = best_group["adducts"]
    primary = adducts[0]
    mol = best_group["mols"][0]
    exact_mass = best_group["exact_mass"]

    # Mass Accuracy Score
    mean_abs_ppm = np.mean([abs(a["ppm"]) for a in adducts])
    ppm_score = max(0.0, 35.0 * (1.0 - (mean_abs_ppm / 15.0)))

    # Dominant m/z Rank
    matched_dom_rel_int = 0.0
    for a in adducts:
        for dom_mz, dom_rel in dom_mz_list:
            if abs(dom_mz - a["calc_mz"]) <= 0.015:
                matched_dom_rel_int = max(matched_dom_rel_int, dom_rel)

    if matched_dom_rel_int >= 95.0:
        dom_rank_score = 35.0
    elif matched_dom_rel_int > 0:
        dom_rank_score = 12.0 + 23.0 * (matched_dom_rel_int / 100.0)
    else:
        dom_rank_score = 5.0

    # Isotope Envelope Verification
    num_C = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)
    num_N = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 7)
    num_S = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 16)

    theo_m1_rel = (num_C * 1.07) + (num_N * 0.37) + (num_S * 0.76)
    if theo_m1_rel == 0:
        theo_m1_rel = 12.0

    m0_mz = primary["calc_mz"]
    m1_target = m0_mz + 1.003355
    m2_target = m0_mz + 2.006710

    iso_score = 0.0
    iso_evidence = []

    m1_obs_rel = None
    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m1_target) <= 0.015 and matched_dom_rel_int > 0:
            m1_obs_rel = (dom_rel / matched_dom_rel_int) * 100.0
            break

    if m1_obs_rel is None and full_window_spectrum and matched_dom_rel_int > 0:
        m0_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m0_mz) <= 0.015], default=0)
        m1_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m1_target) <= 0.015], default=0)
        if m0_int > 0 and m1_int > 0:
            m1_obs_rel = (m1_int / m0_int) * 100.0

    if m1_obs_rel is not None:
        rel_diff = abs(m1_obs_rel - theo_m1_rel)
        iso_score += max(6.0, 16.0 * (1.0 - min(rel_diff / 15.0, 1.0)))
        iso_evidence.append(f"[M+H+1]+ iso ({m1_obs_rel:.0f}% vs {theo_m1_rel:.0f}% theo)")

    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m2_target) <= 0.015:
            iso_score += 4.0
            iso_evidence.append("[M+H+2]+ iso")
            break

    # Multi-Adduct & In-Source Fragments
    corrob_score = 0.0
    if len(adducts) > 1:
        corrob_score += min(7.0, (len(adducts) - 1) * 4.5)

    mh_mz = exact_mass + 1.007276
    for loss_name, shift in insource_neutral_losses:
        frag_mz = mh_mz + shift
        if frag_mz > 80.0:
            for dom_mz, dom_rel in dom_mz_list:
                if abs(dom_mz - frag_mz) <= 0.015:
                    corrob_score += 5.0
                    iso_evidence.append(f"{loss_name} ({dom_mz:.3f})")
                    break

    corrob_score = min(10.0, corrob_score)
    total_conf = round(min(99.9, ppm_score + dom_rank_score + iso_score + corrob_score), 1)

    evidence_summary = f"Dom.Rank: {matched_dom_rel_int:.0f}%"
    if iso_evidence:
        evidence_summary += " | " + ", ".join(iso_evidence[:2])

    return total_conf, evidence_summary

# --- 4. Interactive UI & Unified Execution Pipeline ---
prominence_input = widgets.FloatText(
    value=default_prominence,
    description='Prominence:',
    step=0.001,
    style={'description_width': 'initial'}
)

rerun_button = widgets.Button(
    description='Rerun Peak Fitting & Assignment',
    button_style='primary',
    icon='refresh',
    layout=widgets.Layout(width='240px')
)

output_area = widgets.Output()

# Per-peak, summed MS1 spectrum (m/z -> intensity) captured across each peak's elution
# window. Populated by run_unified_pipeline and consumed by the interactive multi-peak
# comparison widgets below (isotope envelopes, co-eluting isomers, fine mass differences).
peak_ms1_spectra = {}

ms_peak_selector = widgets.SelectMultiple(
    options=[],
    description='Compare Peaks:',
    rows=6,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='460px')
)

ms_output_area = widgets.Output()


def _peak_comparison_label(p_id):
    """Human-readable label for a peak_id, reused by the chromatogram legend/cards and
    the MS1 spectra comparison selector/subplot titles."""
    if p_id in peak_assignments:
        a_info = peak_assignments[p_id]
        iso_short = " [Isobaric Isomers]" if a_info["is_isobaric"] else ""
        adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts']])
        return f"Peak {p_id}: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
    if 'detected_df' in globals():
        dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
        if len(dom_rows):
            dom_str = dom_rows.values[0]
            first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
            return f"Peak {p_id}: Unassigned (Base m/z: {first_ion})"
    return f"Peak {p_id}: Unassigned"


def build_ms_comparison_figure(selected_peak_ids):
    """Builds a stacked, zoom-linked Plotly figure overlaying the MS1 stick spectrum of
    every selected peak so isotope envelopes, co-eluting isomers and fine (sub-0.01 Da)
    mass differences can be inspected side-by-side. All subplot x-axes share a single
    zoom/pan state (via `matches='x'`), so zooming into one isotope envelope zooms every
    other peak's spectrum to the same m/z window simultaneously."""
    mpl_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    n = max(len(selected_peak_ids), 1)

    fig_ms = make_subplots(
        rows=n, cols=1, shared_xaxes=True,
        vertical_spacing=min(0.12, 0.6 / n) if n > 1 else 0.0,
        subplot_titles=[_peak_comparison_label(p) for p in selected_peak_ids]
    )

    for row_i, p_id in enumerate(selected_peak_ids, start=1):
        # Use our helper function to convert MPL color value safely to Plotly
        plotly_color_val = to_plotly_color(mpl_colors[(p_id - 1) % len(mpl_colors)])
        spectrum = peak_ms1_spectra.get(p_id, {})

        if spectrum:
            mzs = np.array(sorted(spectrum.keys()))
            intens = np.array([spectrum[m] for m in mzs])
            max_int = intens.max()
            rel = (intens / max_int * 100.0) if max_int > 0 else intens

            # Interleave (mz, 0) -> (mz, rel) -> (mz, NaN) to render each m/z as a
            # discrete stick (lollipop), the standard way to depict an MS1 spectrum.
            stick_x = np.repeat(mzs, 3)
            stick_y = np.zeros(stick_x.shape, dtype=float)
            stick_y[1::3] = rel
            stick_y[2::3] = np.nan

            fig_ms.add_trace(
                go.Scatter(
                    x=stick_x, y=stick_y, mode='lines',
                    line=dict(color=plotly_color_val, width=1.3),
                    hoverinfo='skip', showlegend=False
                ),
                row=row_i, col=1
            )
            fig_ms.add_trace(
                go.Scatter(
                    x=mzs, y=rel, mode='markers',
                    marker=dict(color=plotly_color_val, size=5),
                    customdata=intens, showlegend=False,
                    hovertemplate=(
                        f"Peak {p_id}<br>m/z=%{{x:.4f}}<br>rel. int.=%{{y:.1f}}%"
                        "<br>abs. int.=%{customdata:.3e}<extra></extra>"
                    )
                ),
                row=row_i, col=1
            )

            # Mark the theoretical (calculated) m/z of the top assigned adducts as dashed
            # reference lines, making fine mass-accuracy (ppm) differences visible at a
            # glance once the user zooms in.
            if p_id in peak_assignments:
                for adduct in peak_assignments[p_id]['adducts'][:3]:
                    fig_ms.add_vline(
                        x=adduct['calc_mz'], line=dict(color=plotly_color_val, width=1.0, dash='dot'),
                        annotation_text=f"{adduct['adduct']} calc.", annotation_font_size=8,
                        annotation_position="bottom", row=row_i, col=1
                    )
        else:
            fig_ms.add_annotation(
                text="No MS1 signal captured in this peak's elution window",
                x=0.5, y=0.5, xref=f'x{row_i} domain', yref=f'y{row_i} domain',
                showarrow=False, font=dict(color='#9ca3af', size=11)
            )

        fig_ms.update_yaxes(title_text="Rel. Int. (%)", range=[0, 120], row=row_i, col=1)

    fig_ms.update_xaxes(matches='x', title_text="m/z", row=n, col=1)
    fig_ms.update_layout(
        height=max(300, 260 * n),
        template='plotly_white',
        title_text="Multi-Peak MS1 Spectra Comparison — Isotope Envelopes, Co-eluting Isomers & Fine Mass Differences",
        margin=dict(t=90, b=50),
        hovermode='closest'
    )
    return fig_ms


def render_ms_comparison(change=None):
    with ms_output_area:
        clear_output(wait=True)
        selected = list(ms_peak_selector.value)
        if not selected:
            print("Select one or more peaks above to overlay/stack their MS1 spectra for comparison.")
            return
        build_ms_comparison_figure(selected).show()


ms_peak_selector.observe(render_ms_comparison, names='value')


def run_unified_pipeline(prom_val):
    global detected_df, peak_assignments
    with output_area:
        clear_output(wait=True)
        print(f"Using prominence threshold: {prom_val:.4f}")

        # Topographical peak detection
        peak_indices, _ = scipy.signal.find_peaks(signal_norm, prominence=prom_val)
        _, left_bases, right_bases = scipy.signal.peak_prominences(signal_norm, peak_indices)

        start_times = hplc_df['Time'].iloc[left_bases].values
        end_times = hplc_df['Time'].iloc[right_bases].values

        print(f"Detected {len(peak_indices)} raw topographical anomalies. Deconvoluting true peaks...")
        chrom = Chromatogram(hplc_df, cols={'time': 'Time', 'signal': 'Signal'}, time_window=[200, 300])
        peaks = chrom.fit_peaks(prominence=prom_val, buffer=0).reset_index(drop=True)

        mapped_starts, mapped_ends, mapped_topos = [], [], []
        peak_rts = hplc_df['Time'].values[peak_indices]

        for rt in peaks['retention_time']:
            if len(peak_rts) > 0:
                closest_idx = np.argmin(np.abs(peak_rts - rt))
                mapped_starts.append(start_times[closest_idx])
                mapped_ends.append(end_times[closest_idx])
                mapped_topos.append(peak_rts[closest_idx])
            else:
                mapped_starts.append(np.nan)
                mapped_ends.append(np.nan)
                mapped_topos.append(np.nan)

        peaks['peak start time'] = mapped_starts
        peaks['peak end time'] = mapped_ends
        peaks['topographical_apex'] = mapped_topos

        if peaks['area'].sum() > 0:
            peaks['normalised area'] = (peaks['area'] / peaks['area'].sum() * 100).map('{:.2f}%'.format)
        else:
            peaks['normalised area'] = '0.00%'

        # Extract top 10 dominant m/z signals
        peak_spectra = {i: {} for i in range(len(peaks))}
        if os.path.exists(mzml_filename):
            with mzml.read(mzml_filename) as reader:
                for spectrum in reader:
                    if spectrum['ms level'] == 1:
                        scan_data = spectrum.get('scanList', {}).get('scan', [{}])
                        rt = scan_data[0].get('scan start time', -1) if isinstance(scan_data, list) and scan_data else (scan_data.get('scan start time', -1) if isinstance(scan_data, dict) else -1)
                        mz_array = spectrum.get('m/z array', [])
                        int_array = spectrum.get('intensity array', [])

                        if len(mz_array) > 0:
                            mzs_rounded = np.round(mz_array, 3)
                            for i, row in peaks.iterrows():
                                if not pd.isna(row['peak start time']) and row['peak start time'] <= rt <= row['peak end time']:
                                    for m, intensity in zip(mzs_rounded, int_array):
                                        peak_spectra[i][m] = peak_spectra[i].get(m, 0) + intensity

        TOP_N_PEAKS = 10
        MIN_REL_INTENSITY = 2.0
        dominant_mz_col = []
        for i in range(len(peaks)):
            spectra = peak_spectra[i]
            if not spectra:
                dominant_mz_col.append("None")
                continue
            sorted_mzs = sorted(spectra.items(), key=lambda x: x[1], reverse=True)
            max_int = sorted_mzs[0][1]
            top_strs = []
            for m, inten in sorted_mzs:
                rel_int = (inten / max_int) * 100
                if rel_int >= MIN_REL_INTENSITY:
                    top_strs.append(f"{m:.3f} ({rel_int:.0f}%)")
                if len(top_strs) >= TOP_N_PEAKS:
                    break
            dominant_mz_col.append(", ".join(top_strs) if top_strs else "None")

        peaks['dominant m/z'] = dominant_mz_col

        # Screen ESI+ adducts & score
        candidates = get_candidates_from_table()
        theoretical_ions = []
        for name, classification, smi in candidates:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                exact_mass = Descriptors.ExactMolWt(mol)
                formula = rdMolDescriptors.CalcMolFormula(mol)
                for adduct_name, mult, shift, z in esi_pos_adducts:
                    calc_mz = (exact_mass * mult + shift) / z
                    theoretical_ions.append({
                        "name": f"{name} ({classification})",
                        "raw_name": name,
                        "classification": classification,
                        "smiles": smi,
                        "mol": mol,
                        "formula": formula,
                        "exact_mass": round(exact_mass, 4),
                        "adduct": adduct_name,
                        "calc_mz": calc_mz
                    })

        PPM_TOLERANCE = 15.0
        MIN_PEAK_INTENSITY = 5e5
        peak_assignments = {}

        if 'od' in globals():
            for idx_row, peak_row in peaks.iterrows():
                p_id = int(peak_row['peak_id'])
                rt_start = peak_row['peak start time']
                rt_end = peak_row['peak end time']
                rt_apex = peak_row['retention_time']
                dom_mz_parsed = parse_dominant_mz_string(peak_row['dominant m/z'])

                if pd.isna(rt_start) or pd.isna(rt_end):
                    continue

                window_scans = [
                    spec for spec in od
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end
                ]
                if not window_scans:
                    continue

                mass_group_hits = {}
                for ion in theoretical_ions:
                    target_mz = ion["calc_mz"]
                    tol_da = target_mz * (PPM_TOLERANCE * 1e-6)

                    max_int = 0.0
                    obs_mz_at_max = target_mz
                    rt_at_max = rt_apex

                    for spec in window_scans:
                        mzs, ints = spec.get_peaks()
                        mask = (mzs >= target_mz - tol_da) & (mzs <= target_mz + tol_da)
                        if np.any(mask):
                            best_idx = np.argmax(ints[mask])
                            peak_int = float(ints[mask][best_idx])
                            if peak_int > max_int:
                                max_int = peak_int
                                obs_mz_at_max = float(mzs[mask][best_idx])
                                rt_at_max = float(spec.getRT())

                    if max_int >= MIN_PEAK_INTENSITY and abs(rt_at_max - rt_apex) <= 6.0:
                        em = ion["exact_mass"]
                        if em not in mass_group_hits:
                            mass_group_hits[em] = {
                                "exact_mass": em,
                                "formula": ion["formula"],
                                "names": [],
                                "raw_names": [],
                                "mols": [],
                                "total_intensity": 0.0,
                                "adducts": {}
                            }
                        if ion["name"] not in mass_group_hits[em]["names"]:
                            mass_group_hits[em]["names"].append(ion["name"])
                            mass_group_hits[em]["raw_names"].append(ion["raw_name"])
                            mass_group_hits[em]["mols"].append(ion["mol"])

                        if ion["adduct"] not in mass_group_hits[em]["adducts"]:
                            ppm_err = ((obs_mz_at_max - target_mz) / target_mz) * 1e6
                            mass_group_hits[em]["adducts"].setdefault(ion["adduct"], {
                                "adduct": ion["adduct"],
                                "obs_mz": obs_mz_at_max,
                                "calc_mz": target_mz,
                                "ppm": ppm_err,
                                "intensity": max_int
                            })
                            mass_group_hits[em]["total_intensity"] += max_int

                if mass_group_hits:
                    best_group = max(mass_group_hits.values(), key=lambda x: x["total_intensity"])
                    sorted_adducts = sorted(best_group["adducts"].values(), key=lambda x: x["intensity"], reverse=True)
                    best_group["adducts"] = sorted_adducts

                    is_isobaric = len(best_group["raw_names"]) > 1
                    iso_tag = f" [Isobaric Isomers, {best_group['formula']}]" if is_isobaric else ""

                    conf_score, evidence_str = compute_confidence_from_dominant_mz(
                        best_group, dom_mz_parsed, peak_spectra.get(idx_row, {})
                    )

                    peak_assignments[p_id] = {
                        "display_name": " / ".join(best_group["raw_names"]),
                        "full_assignment": " / ".join(best_group["names"]) + iso_tag,
                        "is_isobaric": is_isobaric,
                        "formula": best_group["formula"],
                        "mols": best_group["mols"],
                        "adducts": sorted_adducts,
                        "confidence": conf_score,
                        "evidence": evidence_str
                    }

        assignment_col = []
        confidence_col = []
        for idx_row, row in peaks.iterrows():
            p_id = int(row['peak_id'])
            if p_id in peak_assignments:
                assign_info = peak_assignments[p_id]
                top_adduct = assign_info["adducts"][0]
                assignment_col.append(
                    f"{assign_info['full_assignment']} [{top_adduct['adduct']}: {top_adduct['obs_mz']:.4f} m/z ({top_adduct['ppm']:+.1f} ppm)]"
                )
                confidence_col.append(f"{assign_info['confidence']:.1f}% ({assign_info['evidence']})")
            else:
                dom_list = parse_dominant_mz_string(row['dominant m/z'])
                base_str = f"Base m/z: {dom_list[0][0]:.3f}" if dom_list else "No MS1 signal"
                assignment_col.append(f"Unassigned ({base_str})")
                confidence_col.append("N/A")

        peaks['Assignment'] = assignment_col
        peaks['Confidence (%)'] = confidence_col
        detected_df = peaks.copy()

        def highlight_peaks(row):
            colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
            c = mcolors.to_rgba(colors[(int(row['peak_id']) - 1) % len(colors)], alpha=0.4)
            bg_color = f'rgba({int(c[0]*255)}, {int(c[1]*255)}, {int(c[2]*255)}, {c[3]})'
            return [f'background-color: {bg_color}' for _ in row]

        display(detected_df.style.apply(highlight_peaks, axis=1))
        print(f"\nFinal deconvoluted peak count: {len(detected_df)} | Assigned peaks: {len(peak_assignments)}")

        # E. Render Interactive, Zoomable Plotly Chromatogram with Structure Cards
        FIG_WIDTH_PX, FIG_HEIGHT_PX = 1150, 760
        PLOT_AREA_WIDTH_PX, PLOT_AREA_HEIGHT_PX = FIG_WIDTH_PX - 330, FIG_HEIGHT_PX - 160

        time_arr = chrom.df[chrom.time_col].values
        raw_signal = chrom.df[chrom.int_col].values
        convolved = np.sum(chrom.unmixed_chromatograms, axis=1) if chrom.peaks is not None else None
        y_max_plot = float(np.max(raw_signal))

        colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

        # Staggered offsets explicitly separated to guarantee zero image card overlap
        assigned_card_offsets = {
            1: (204.5, y_max_plot * 0.50),
            2: (218.5, y_max_plot * 1.25),
            3: (236.0, y_max_plot * 0.85),
            4: (258.0, y_max_plot * 1.25),
            5: (275.0, y_max_plot * 0.80),
        }

        chrom_fig = go.Figure()
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=raw_signal, mode='lines', name='raw chromatogram',
            line=dict(color='black', width=1.6),
            hovertemplate='RT=%{x:.2f}s<br>Signal=%{y:.4f}<extra></extra>'
        ))
        if convolved is not None:
            chrom_fig.add_trace(go.Scatter(
                x=time_arr, y=convolved, mode='lines', name='inferred mixture',
                line=dict(color='red', width=1.4, dash='dash'),
                hovertemplate='RT=%{x:.2f}s<br>Mixture=%{y:.4f}<extra></extra>'
            ))

        if chrom.peaks is not None:
            for g, _ in chrom.peaks.groupby('peak_id'):
                p_id = int(g)
                peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
                component = chrom.unmixed_chromatograms[:, p_id - 1]
                legend_label = _peak_comparison_label(p_id)
                chrom_fig.add_trace(go.Scatter(
                    x=time_arr, y=component, mode='lines', fill='tozeroy',
                    line=dict(color=peak_color, width=0.5), opacity=0.5,
                    name=legend_label,
                    hovertemplate=f'{legend_label}<br>RT=%{{x:.2f}}s<br>Signal=%{{y:.4f}}<extra></extra>'
                ))

        shapes, images, annotations = [], [], []

        for _, peak_row in detected_df.iterrows():
            p_id = int(peak_row['peak_id'])
            rt_apex = float(peak_row['retention_time'])
            c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
            apex_y = float(raw_signal[c_idx])
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

            # Drop-line for the apex
            shapes.append(dict(
                type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y,
                line=dict(color='dodgerblue', width=1.2, dash='dot'), opacity=0.6
            ))

            if p_id in peak_assignments:
                assign = peak_assignments[p_id]
                mol_images = [Draw.MolToImage(m, size=(160, 140)) for m in assign["mols"]]
                if len(mol_images) > 1:
                    total_w = sum(im.width for im in mol_images)
                    struct_img = Image.new('RGBA', (total_w, mol_images[0].height), (255, 255, 255, 255))
                    x_off = 0
                    for im in mol_images:
                        struct_img.paste(im, (x_off, 0))
                        x_off += im.width
                else:
                    struct_img = mol_images[0]

                adduct_lines = "<br>".join([
                    f"{a['adduct']}: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                    for a in assign["adducts"]
                ])
                iso_header = f"<br>⚠️ Isobaric Isomers ({assign['formula']})" if assign["is_isobaric"] else ""
                header_text = (
                    f"<b>Peak {p_id} ({rt_apex:.1f}s): {assign['display_name']}</b>"
                    + iso_header + "<br>" + adduct_lines
                    + f"<br>Confidence: {assign['confidence']:.1f}%"
                )

                box_x, box_y = assigned_card_offsets.get(p_id, (rt_apex, apex_y + y_max_plot * 0.45))

                # Arrow connecting the apex to the floating card
                annotations.append(dict(
                    x=rt_apex, y=apex_y, ax=box_x, ay=box_y, xref='x', yref='y', axref='x', ayref='y',
                    showarrow=True, arrowhead=2, arrowcolor=peak_color, arrowwidth=1.8, text=''
                ))
                # Text card (assignment, adducts, confidence)
                annotations.append(dict(
                    x=box_x, y=box_y + y_max_plot * 0.085, xref='x', yref='y',
                    text=header_text, showarrow=False, align='center',
                    font=dict(size=10.5, color='#111827'),
                    bgcolor='white', bordercolor=peak_color, borderwidth=2, borderpad=4
                ))
                # Structure image, sized to preserve pixel aspect ratio on the data axes
                px_per_xunit = PLOT_AREA_WIDTH_PX / 120.0
                px_per_yunit = PLOT_AREA_HEIGHT_PX / (y_max_plot * 1.58)
                images.append(dict(
                    source=struct_img, xref='x', yref='y',
                    x=box_x, y=box_y - y_max_plot * 0.02,
                    sizex=struct_img.width / px_per_xunit, sizey=struct_img.height / px_per_yunit,
                    xanchor='center', yanchor='top', layer='above'
                ))
            else:
                dom_str = str(peak_row.get('dominant m/z', 'None'))
                dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
                top_ions_str = ", ".join(dom_tokens[:2]) if dom_tokens else "N/A"
                box_y = apex_y + y_max_plot * 0.15

                annotations.append(dict(
                    x=rt_apex, y=apex_y, ax=rt_apex, ay=box_y, xref='x', yref='y', axref='x', ayref='y',
                    showarrow=True, arrowhead=2, arrowcolor=peak_color, arrowwidth=1.2,
                    text=f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned<br>Dom. m/z: {top_ions_str}",
                    font=dict(size=9, color='#374151'),
                    bgcolor='#f9fafb', bordercolor=peak_color, borderwidth=1.3
                ))

        chrom_fig.update_layout(
            shapes=shapes, images=images, annotations=annotations,
            title=dict(
                text=f"HPLC Chromatogram Deconvolution, Isobaric Screening & Dominant-m/z Confidence (Prominence: {prom_val:.4f})",
                font=dict(size=16, color='#111827')
            ),
            xaxis=dict(title='Retention Time (seconds)', range=[190, 310], rangeslider=dict(visible=True)),
            yaxis=dict(title='Total Ion Count (baseline corrected)', range=[0, y_max_plot * 1.58]),
            legend=dict(x=1.02, y=1, xanchor='left', bgcolor='white', bordercolor='#d1d5db', borderwidth=1),
            template='plotly_white', width=FIG_WIDTH_PX, height=FIG_HEIGHT_PX, hovermode='closest'
        )
        chrom_fig.show()

        # Cache a PNG snapshot — rendered via kaleido from the exact interactive figure
        # above — so the multimodal vision QC cell can still inspect the same chromatogram
        # the user is zooming/panning through, even though it is no longer a static
        # matplotlib Figure.
        try:
            chromatogram_png_bytes = chrom_fig.to_image(format='png', scale=2)
            globals().setdefault('_notebook_figure_registry', {})['deconvoluted_chromatogram_with_structures'] = chromatogram_png_bytes
        except Exception as e:
            print(f"⚠️ Could not rasterize interactive chromatogram for vision QC (kaleido missing?): {e}")

        globals()['_last_chromatogram_figure'] = chrom_fig

        # F. Populate the per-peak MS1 spectra store & refresh the multi-peak comparison widget
        global peak_ms1_spectra
        peak_ms1_spectra = {int(row['peak_id']): peak_spectra.get(idx_row, {}) for idx_row, row in peaks.iterrows()}

        ms_peak_selector.options = [
            (_peak_comparison_label(p_id), p_id) for p_id in sorted(peak_ms1_spectra.keys())
        ]
        preselected = tuple(p_id for _, p_id in ms_peak_selector.options[:3])
        if ms_peak_selector.value == preselected:
            render_ms_comparison()
        else:
            ms_peak_selector.value = preselected

        display(widgets.HTML(
            "<b>🔬 Multi-Peak MS1 Spectra Comparison</b> — select 2+ peaks above to overlay/stack their "
            "MS1 stick spectra on a shared, zoomable m/z axis and inspect isotope envelopes, co-eluting "
            "isomers, and fine mass differences side-by-side."
        ))
        display(ms_peak_selector)
        display(ms_output_area)

def on_rerun_clicked(b):
    run_unified_pipeline(prominence_input.value)

rerun_button.on_click(on_rerun_clicked)

display(widgets.HBox([prominence_input, rerun_button]))
display(output_area)

run_unified_pipeline(default_prominence)

In [ ]:
import plotly.io as pio
import sys

# Ensure Plotly renders correctly in Colab
if 'google.colab' in sys.modules:
    pio.renderers.default = 'colab'

# Retrieve and display the cached interactive chromatogram
if '_last_chromatogram_figure' in globals():
    print("Rendered Interactive Chromatogram (Safe Mode):")
    _last_chromatogram_figure.show()
else:
    print("No cached chromatogram found. Please make sure the deconvolution cell above finishes running first.")



First, add a new code cell to install the official Google GenAI SDK, the MCP client, and the RDKit MCP server package:

In [ ]:
#@title 1. Install the MCP Bridge and Server

!uv pip install --system -q google-genai mcp rdkit

Since spinning up a separate JSON-RPC subprocess server inside a transient Colab cell can sometimes fight with Colab's I/O structure, the most robust way to give Gemini "MCP-like" tool-use capabilities in a notebook is to define the tools explicitly using Gemini's native function-calling API (google-genai SDK):

In [ ]:
#@title Step 2: Self-Healing RDKit Tools, Automated Fragmenter & 2D SVG Generator
import re
import base64
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from rdkit.Chem.Draw import rdMolDraw2D

# 1. Silence noisy C++ stderr logs from RDKit so Colab output stays clean
RDLogger.DisableLog('rdApp.*')

_rendered_svg_registry = {}

def safe_mol_from_smiles(smiles_or_str: str):
    """
    Robust SMILES parser that automatically repairs common LLM kekulization mistakes
    (e.g., missing [nH] in cleaved purines/indoles/imidazoles like 'Nc1ncnc2c1ncn2').
    Returns (mol, canonical_smiles, error_msg).
    """
    s = smiles_or_str.strip()
    if not s:
        return None, None, "Empty input string."

    # Check if the model accidentally passed a Molecular Formula (e.g., 'C10H13N5O4')
    if re.fullmatch(r'([A-Z][a-z]?\d*)+([+-]\d*)?', s) and not any(c in s for c in ['c', 'n', 'o', 's', '=', '#', '(', ')', '[']):
        if any(char.isdigit() for char in s) and not re.search(r'[1-9].*[1-9]', s[:3]):
            return None, None, (
                f"'{s}' is a molecular formula, not a SMILES string. "
                "Use `calculate_formula_mass` for formulas, or pass a valid SMILES (e.g. 'Nc1ncnc2c1nc[nH]2' for adenine)."
            )

    # Attempt 1: Standard RDKit parse
    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol, Chem.CanonSmiles(s), None

    # Attempt 2: Auto-repair unkekulized pyrrole/purine/imidazole nitrogens (missing [nH])
    mol_no_san = Chem.MolFromSmiles(s, sanitize=False)
    if mol_no_san is not None:
        try:
            mol_no_san.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol_no_san)
            for atom in mol_no_san.GetAtoms():
                # Aromatic N with degree 2, 0 formal charge, and 0 explicit H often needs 1 H in 5-membered rings
                if atom.GetAtomicNum() == 7 and atom.GetIsAromatic() and atom.GetDegree() == 2 and atom.GetFormalCharge() == 0 and atom.GetNumExplicitHs() == 0:
                    test_mol = Chem.Mol(mol_no_san)
                    test_mol.GetAtomWithIdx(atom.GetIdx()).SetNumExplicitHs(1)
                    try:
                        Chem.SanitizeMol(test_mol)
                        fixed_smi = Chem.MolToSmiles(test_mol)
                        return test_mol, fixed_smi, None
                    except Exception:
                        continue
        except Exception:
            pass

    return None, None, (
        f"RDKit could not parse or kekulize '{s}'. "
        "Tip: In aromatic 5-membered rings (purines, adenine, histidine, indoles), "
        "explicitly write '[nH]' for the protonated ring nitrogen (e.g., adenine is 'Nc1ncnc2c1nc[nH]2')."
    )


def get_user_candidate_molecules() -> str:
    """Returns the user's input table of candidate molecules, classifications, SMILES, formulas, and exact masses."""
    candidates = get_candidates_from_table() if 'get_candidates_from_table' in globals() else []
    if not candidates:
        return "No candidate molecules found."

    rows = []
    for name, classification, smi in candidates:
        mol, clean_smi, _ = safe_mol_from_smiles(smi)
        if mol:
            em = Descriptors.ExactMolWt(mol)
            formula = rdMolDescriptors.CalcMolFormula(mol)
            rows.append({
                "Name": name,
                "Classification": classification,
                "SMILES": clean_smi,
                "Formula": formula,
                "Exact_Mass": round(em, 4),
                "[M+H]+_mz": round(em + 1.007276, 4)
            })
    return pd.DataFrame(rows).to_markdown(index=False)


def get_detected_peaks_summary() -> str:
    """Returns a full tabular summary of all deconvoluted HPLC-MS peaks, retention times, areas, dominant m/z ions, and assignments."""
    if 'detected_df' not in globals():
        return "Error: detected_df not found in memory. Run the peak detection cell first."

    cols = [c for c in ['peak_id', 'retention_time', 'peak start time', 'peak end time', 'normalised area', 'dominant m/z', 'Assignment', 'Confidence (%)'] if c in detected_df.columns]
    return detected_df[cols].to_markdown(index=False)


def get_peak_details(peak_id: int) -> dict:
    """Retrieves detailed analytical evidence, observed adducts, and raw MS1 ions for a specific peak ID."""
    result = {}
    if 'detected_df' in globals():
        row = detected_df[detected_df['peak_id'] == peak_id]
        if not row.empty:
            r_dict = row.iloc[0].to_dict()
            result["chromatogram_row"] = {k: str(v) for k, v in r_dict.items()}
            # Also query raw spectrum across this peak's start and end window if 'od' is loaded
            rt_start = r_dict.get('peak start time', np.nan)
            rt_end = r_dict.get('peak end time', np.nan)
            if 'od' in globals() and not pd.isna(rt_start) and not pd.isna(rt_end):
                agg = {}
                for spec in od:
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
                        mzs, ints = spec.get_peaks()
                        for m, inten in zip(np.round(mzs, 4), ints):
                            if inten > 1e3:
                                agg[m] = agg.get(m, 0.0) + float(inten)
                if agg:
                    top_ions = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:25]
                    max_i = top_ions[0][1]
                    result["raw_ms1_top_25_ions_in_peak_window"] = [
                        {"mz": float(m), "intensity": round(i, 1), "rel_pct": round((i / max_i) * 100, 2)}
                        for m, i in top_ions
                    ]

    if 'peak_assignments' in globals() and peak_id in peak_assignments:
        assign = peak_assignments[peak_id]
        result["assigned_compound"] = {
            "display_name": assign["display_name"],
            "formula": assign["formula"],
            "confidence": assign["confidence"],
            "evidence": assign["evidence"],
            "adducts_observed": [
                f"{a['adduct']} at {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm, intensity: {a['intensity']:.1e})"
                for a in assign["adducts"]
            ]
        }
    return result if result else {"error": f"Peak {peak_id} not found."}


def check_mz_in_peak(peak_id: int, target_mz: float, tolerance_da: float = 0.05) -> dict:
    """
    Directly searches all raw MS1 scans across a specific peak's retention time window
    to verify whether a specific m/z (e.g. a fragment ion) is present in that peak.
    """
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found in memory."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])
    dom_str = str(row.iloc[0]['dominant m/z'])

    matches = []
    max_scan_base_peak = 1.0
    for spec in od:
        if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
            mzs, ints = spec.get_peaks()
            if len(ints) > 0:
                max_scan_base_peak = max(max_scan_base_peak, float(np.max(ints)))
            mask = (mzs >= target_mz - tolerance_da) & (mzs <= target_mz + tolerance_da)
            if np.any(mask):
                for m, inten in zip(mzs[mask], ints[mask]):
                    matches.append({
                        "rt": round(float(spec.getRT()), 2),
                        "observed_mz": round(float(m), 4),
                        "error_da": round(float(m - target_mz), 4),
                        "ppm_error": round(float((m - target_mz) / target_mz * 1e6), 1),
                        "intensity": round(float(inten), 1)
                    })

    matches = sorted(matches, key=lambda x: x["intensity"], reverse=True)
    return {
        "peak_id": peak_id,
        "rt_window": f"{rt_start:.1f}s - {rt_end:.1f}s",
        "dominant_mz_summary": dom_str,
        "target_mz_searched": target_mz,
        "tolerance_da": tolerance_da,
        "found_in_raw_scans": len(matches) > 0,
        "max_observed_hit": matches[0] if matches else None,
        "top_5_scan_hits": matches[:5]
    }


def generate_in_silico_fragments(parent_smiles: str, check_peak_id: int = 0) -> dict:
    """
    Automatically generates chemically valid, RDKit-sanitized ESI+ fragments for a parent molecule
    (including glycosidic cleavage, neutral losses of H2O/NH3/CO, and single acyclic bond cleavages).
    Optionally checks if each fragment m/z is observed in `check_peak_id`.
    """
    mol, clean_parent, err = safe_mol_from_smiles(parent_smiles)
    if not mol:
        return {"error": err}

    parent_em = Descriptors.ExactMolWt(mol)
    parent_formula = rdMolDescriptors.CalcMolFormula(mol)
    parent_mh = round(parent_em + 1.007276, 4)

    fragments_dict = {}

    def _add_frag(frag_smi, pathway_label, is_intrinsic_cation=False):
        f_mol, f_clean, _ = safe_mol_from_smiles(frag_smi)
        if not f_mol:
            return
        em = Descriptors.ExactMolWt(f_mol)
        if em < 40.0 or em >= parent_em - 1.0:
            return
        form = rdMolDescriptors.CalcMolFormula(f_mol)
        f_charge = Chem.GetFormalCharge(f_mol)
        if is_intrinsic_cation or f_charge > 0:
            mz_val = round(em - 0.00054858, 4)
            ion_type = "[F]+ (oxocarbenium / intrinsic cation)"
        else:
            mz_val = round(em + 1.007276, 4)
            ion_type = "[F+H]+"

        key = (form, round(mz_val, 3))
        if key not in fragments_dict:
            entry = {
                "fragment_name": pathway_label,
                "smiles": f_clean,
                "formula": form,
                "neutral_exact_mass": round(em, 4),
                "ion_type": ion_type,
                "expected_esi_pos_mz": mz_val,
                "neutral_loss_from_parent_Da": round(parent_mh - mz_val, 4)
            }
            if check_peak_id > 0:
                hit_info = check_mz_in_peak(check_peak_id, mz_val, tolerance_da=0.05)
                entry["present_in_peak"] = hit_info.get("found_in_raw_scans", False)
                entry["peak_observation"] = hit_info.get("max_observed_hit", None)
            fragments_dict[key] = entry

    # 1. Check canonical nucleoside / riboside pathways if ribose/deoxyribose is present
    ribose_smarts = Chem.MolFromSmarts("[C@@H,C@H,CH]1O[C@H,C@@H,CH](CO)[C@@H,C@H,CH](O)[C@H,C@@H,CH]1")
    if mol.HasSubstructMatch(ribose_smarts):
        # Classic ribose oxocarbenium ion [C5H7O3]+ and dehydrated derivatives
        _add_frag("OC[C@H]1O[C+]=[C@H][C@@H]1O", "Ribose oxocarbenium fragment [C5H7O3]+", is_intrinsic_cation=True)
        _add_frag("O=CC1=CC(=O)CC1", "Dehydrated ribose fragment (C5H5O2+)", is_intrinsic_cation=False)

    # 2. Cleave all single non-ring bonds using RDKit FragmentOnBonds
    cleavable_bonds = []
    for bond in mol.GetBonds():
        if not bond.IsInRing() and bond.GetBondType() == Chem.BondType.SINGLE:
            cleavable_bonds.append(bond.GetIdx())

    for b_idx in cleavable_bonds:
        try:
            frag_mol = Chem.FragmentOnBonds(mol, [b_idx], addDummies=False)
            frags = Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False)
            for f in frags:
                raw_smi = Chem.MolToSmiles(f)
                _add_frag(raw_smi, "Single-bond cleavage fragment ([F+H]+)")
        except Exception:
            pass

    # 3. Common nucleoside cross-ring and base-loss pathways
    if "ncnc2c1ncn2" in clean_parent or "ncnc2c1nc[nH]2" in clean_parent:
        _add_frag("Nc1ncnc2c1nc[nH]2", "Protonated Adenine [B+H]+ (glycosidic cleavage, -Ribose -132.04 Da)")
        _add_frag("c1ncnc2c1nc[nH]2", "Purine core after NH3 loss from Adenine ([B+H-NH3]+, m/z 120.04)")
        _add_frag("Nc1ncnc2c1ncn2C=C", "0,2X Cross-ring cleavage (Adenine + C2H2O/vinyl remnant)")
        _add_frag("Nc1ncnc2c1ncn2CC=O", "0,2X Cross-ring ribose cleavage (Adenine + C2H3O, [M+H-C3H6O3]+)")
        _add_frag("Nc1ncnc2c1ncn2[C@@H](O)C=O", "0,3X Cross-ring ribose cleavage (Adenine + C2H3O2, [M+H-C3H6O2]+)")

    sorted_frags = sorted(fragments_dict.values(), key=lambda x: x["expected_esi_pos_mz"], reverse=True)
    return {
        "parent_smiles": clean_parent,
        "parent_formula": parent_formula,
        "parent_exact_mass": round(parent_em, 4),
        "parent_[M+H]+_mz": parent_mh,
        "generated_fragments": sorted_frags
    }


def analyze_structure_and_mass(smiles_or_formula: str, target_mz: float = 0.0) -> dict:
    """
    Calculates exact monoisotopic mass, molecular formula, and ESI m/z values ([M]+, [M+H]+, [M+Na]+)
    from EITHER a valid SMILES string OR a molecular formula (e.g., 'C5H5N5').
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles_or_formula)
    if mol is None:
        # Try evaluating as a chemical formula directly so it never crashes
        try:
            em = rdMolDescriptors.CalcExactMolWt(smiles_or_formula.strip())
            return {
                "input_interpreted_as": "Molecular Formula (no 2D topology)",
                "formula": smiles_or_formula.strip(),
                "exact_monoisotopic_mass": round(em, 5),
                "calculated_mz_species": {
                    "intrinsic_or_radical_[M]+": round(em - 0.00055, 4),
                    "protonated_[M+H]+": round(em + 1.007276, 4),
                    "sodiated_[M+Na]+": round(em + 22.989218, 4)
                },
                "note": err
            }
        except Exception:
            return {"error": err}

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    formal_charge = Chem.GetFormalCharge(mol)

    mz_intrinsic = exact_mass - (formal_charge * 0.00054858) if formal_charge != 0 else exact_mass
    mz_mh_plus = exact_mass + 1.007276 if formal_charge == 0 else mz_intrinsic

    res = {
        "canonical_smiles": clean_smi,
        "formula": formula,
        "formal_charge": formal_charge,
        "exact_monoisotopic_mass": round(exact_mass, 5),
        "calculated_mz_species": {
            "intrinsic_or_radical_[M]+": round(exact_mass - 0.00055, 4),
            "protonated_[M+H]+": round(mz_mh_plus, 4),
            "sodiated_[M+Na]+": round(exact_mass + 22.989218, 4) if formal_charge == 0 else "N/A"
        }
    }
    if target_mz > 0:
        diff_da = mz_mh_plus - target_mz
        res["target_mz_comparison"] = {
            "target_mz": target_mz,
            "error_Da": round(diff_da, 4),
            "ppm_error": round((diff_da / target_mz) * 1e6, 1)
        }
    return res


def generate_2d_structure_svg(smiles: str, label: str = "", width: int = 280, height: int = 210) -> dict:
    """
    Uses RDKit to generate a 2D chemical structure SVG from a SMILES string.
    Stores the SVG in memory and returns a short placeholder token (e.g. '{{SVG_1}}')
    that you place in your response where the image should appear.
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if not mol:
        return {"error": err}

    AllChem.Compute2DCoords(mol)
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addStereoAnnotation = True
    opts.padding = 0.12

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    mh_mz = exact_mass + 1.007276 if Chem.GetFormalCharge(mol) == 0 else exact_mass
    legend_text = f"{label} | {formula} ([M+H]+: {mh_mz:.4f})" if label else f"{formula} (Exact: {exact_mass:.4f})"

    drawer.DrawMolecule(mol, legend=legend_text)
    drawer.FinishDrawing()
    svg_str = drawer.GetDrawingText()

    # Strip XML header so it embeds cleanly inline in HTML
    svg_clean = re.sub(r"<\?xml.*?\?>", "", svg_str, flags=re.DOTALL).strip()

    token_id = f"{{{{SVG_{len(_rendered_svg_registry) + 1}}}}}"
    _rendered_svg_registry[token_id] = (
        f"<div style='display:inline-block; margin:8px; padding:8px; border:1px solid #d1d5db; "
        f"border-radius:8px; background:white; text-align:center; box-shadow: 0 1px 3px rgba(0,0,0,0.08);'>"
        f"{svg_clean}</div>"
    )

    return {
        "status": "success",
        "canonical_smiles_drawn": clean_smi,
        "formula": formula,
        "exact_mass": round(exact_mass, 4),
        "expected_mh_mz": round(mh_mz, 4),
        "svg_placeholder_token": token_id,
        "instruction": f"Insert the exact token {token_id} on its own line in your response where you want this 2D structure displayed. Do NOT write raw SVG or base64."
    }

By providing these specific functions, analytical chemists can ask Gemini complex, multi-step questions directly in the chat interface, such as:

> "What is the confidence score for the peak containing a pyrimidine ring?" (Gemini will call find_peaks_by_substructure, then get_peak_details).


>  "If I spiked this sample with Caffeine (CN1C=NC2=C1C(=O)N(C(=O)N2C)C), what [M+Na]+ m/z would I look for?" (Gemini will call calculate_expected_adduct_mz).


You can pass these RDKit functions directly to the Gemini API so the model can autonomously invoke them when answering your natural language cheminformatics questions:

In [ ]:
#@title Step 3 (Complete Self-Contained Script): RDKit Tools + Guaranteed 2D Molecule Image Rendering

import re
import base64
import numpy as np
import pandas as pd
from google.colab import userdata
from google import genai
from google.genai import types
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from rdkit.Chem.Draw import rdMolDraw2D

# 1. Silence RDKit C++ warnings
RDLogger.DisableLog('rdApp.*')

# Global dictionary mapping placeholder tokens (e.g. '[[MOL_IMG_1]]' and '{{SVG_1}}') to HTML <img> cards
_rendered_svg_registry = {}

def _register_mol_svg(mol, label: str = "", width: int = 300, height: int = 220) -> str:
    """
    Internal helper: renders an RDKit Mol object to a Base64-encoded SVG <img> HTML card,
    stores it in `_rendered_svg_registry` under both `[[MOL_IMG_N]]` and `{{SVG_N}}`,
    and returns the token `[[MOL_IMG_N]]`.
    """
    mol_copy = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol_copy)

    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addStereoAnnotation = True
    opts.padding = 0.12

    exact_mass = Descriptors.ExactMolWt(mol_copy)
    formula = rdMolDescriptors.CalcMolFormula(mol_copy)
    f_charge = Chem.GetFormalCharge(mol_copy)
    mh_mz = exact_mass + 1.007276 if f_charge == 0 else exact_mass - (f_charge * 0.00054858)

    drawer.DrawMolecule(mol_copy)
    drawer.FinishDrawing()
    svg_str = drawer.GetDrawingText()

    # Encode SVG as base64 inside an HTML <img> tag (renders reliably in Colab dark & light mode)
    b64_svg = base64.b64encode(svg_str.encode('utf-8')).decode('utf-8')
    idx = (len(_rendered_svg_registry) // 2) + 1

    caption_title = f"<div style='font-weight:700; font-size:12px; color:#111827; margin-bottom:4px;'>{label}</div>" if label else ""
    caption_sub = f"<div style='font-size:11px; color:#374151; margin-top:4px;'><b>{formula}</b> | Exact: {exact_mass:.4f} | m/z: {mh_mz:.4f}</div>"

    html_card = (
        f"<div style='display:inline-block; margin:10px 6px; padding:10px 14px; border:2px solid #cbd5e1; "
        f"border-radius:10px; background:#ffffff; text-align:center; box-shadow:0 2px 6px rgba(0,0,0,0.15);'>"
        f"{caption_title}"
        f"<img src='data:image/svg+xml;base64,{b64_svg}' width='{width}' height='{height}' style='display:block; margin:0 auto; background:#ffffff;'/>"
        f"{caption_sub}"
        f"</div>"
    )

    token_primary = f"[[MOL_IMG_{idx}]]"
    token_alt = f"{{{{SVG_{idx}}}}}"
    _rendered_svg_registry[token_primary] = html_card
    _rendered_svg_registry[token_alt] = html_card
    return token_primary


def safe_mol_from_smiles(smiles_or_str: str):
    """Auto-repairs unkekulized pyrrole/purine/imidazole nitrogens (missing [nH])."""
    s = smiles_or_str.strip()
    if not s:
        return None, None, "Empty input string."

    if re.fullmatch(r'([A-Z][a-z]?\d*)+([+-]\d*)?', s) and not any(c in s for c in ['c', 'n', 'o', 's', '=', '#', '(', ')', '[']):
        if any(char.isdigit() for char in s) and not re.search(r'[1-9].*[1-9]', s[:3]):
            return None, None, f"'{s}' is a molecular formula, not a SMILES string."

    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol, Chem.CanonSmiles(s), None

    mol_no_san = Chem.MolFromSmiles(s, sanitize=False)
    if mol_no_san is not None:
        try:
            mol_no_san.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol_no_san)
            for atom in mol_no_san.GetAtoms():
                if atom.GetAtomicNum() == 7 and atom.GetIsAromatic() and atom.GetDegree() == 2 and atom.GetFormalCharge() == 0 and atom.GetNumExplicitHs() == 0:
                    test_mol = Chem.Mol(mol_no_san)
                    test_mol.GetAtomWithIdx(atom.GetIdx()).SetNumExplicitHs(1)
                    try:
                        Chem.SanitizeMol(test_mol)
                        return test_mol, Chem.MolToSmiles(test_mol), None
                    except Exception:
                        continue
        except Exception:
            pass

    return None, None, f"RDKit could not parse '{s}'. Ensure aromatic 5-membered ring NH groups are written as '[nH]'."


# --- RDKIT & LC-MS TOOLS FOR GEMINI ---

def get_user_candidate_molecules() -> str:
    """Returns the user's input table of candidate molecules, classifications, SMILES, formulas, and exact masses."""
    candidates = get_candidates_from_table() if 'get_candidates_from_table' in globals() else []
    if not candidates:
        return "No candidate molecules found."

    rows = []
    for name, classification, smi in candidates:
        mol, clean_smi, _ = safe_mol_from_smiles(smi)
        if mol:
            em = Descriptors.ExactMolWt(mol)
            formula = rdMolDescriptors.CalcMolFormula(mol)
            rows.append({
                "Name": name,
                "Classification": classification,
                "SMILES": clean_smi,
                "Formula": formula,
                "Exact_Mass": round(em, 4),
                "[M+H]+_mz": round(em + 1.007276, 4)
            })
    return pd.DataFrame(rows).to_markdown(index=False)


def get_detected_peaks_summary() -> str:
    """Returns a full tabular summary of all deconvoluted HPLC-MS peaks, retention times, areas, dominant m/z ions, and assignments."""
    if 'detected_df' not in globals():
        return "Error: detected_df not found in memory."
    cols = [c for c in ['peak_id', 'retention_time', 'peak start time', 'peak end time', 'normalised area', 'dominant m/z', 'Assignment', 'Confidence (%)'] if c in detected_df.columns]
    return detected_df[cols].to_markdown(index=False)


def get_peak_details(peak_id: int) -> dict:
    """Retrieves detailed analytical evidence, observed adducts, and raw MS1 ions for a specific peak ID."""
    result = {}
    if 'detected_df' in globals():
        row = detected_df[detected_df['peak_id'] == peak_id]
        if not row.empty:
            r_dict = row.iloc[0].to_dict()
            result["chromatogram_row"] = {k: str(v) for k, v in r_dict.items()}
            rt_start = r_dict.get('peak start time', np.nan)
            rt_end = r_dict.get('peak end time', np.nan)
            if 'od' in globals() and not pd.isna(rt_start) and not pd.isna(rt_end):
                agg = {}
                for spec in od:
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
                        mzs, ints = spec.get_peaks()
                        for m, inten in zip(np.round(mzs, 4), ints):
                            if inten > 1e3:
                                agg[m] = agg.get(m, 0.0) + float(inten)
                if agg:
                    top_ions = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:25]
                    max_i = top_ions[0][1]
                    result["raw_ms1_top_25_ions_in_peak_window"] = [
                        {"mz": float(m), "intensity": round(i, 1), "rel_pct": round((i / max_i) * 100, 2)}
                        for m, i in top_ions
                    ]

    if 'peak_assignments' in globals() and peak_id in peak_assignments:
        assign = peak_assignments[peak_id]
        result["assigned_compound"] = {
            "display_name": assign["display_name"],
            "formula": assign["formula"],
            "confidence": assign["confidence"],
            "evidence": assign["evidence"],
            "adducts_observed": [
                f"{a['adduct']} at {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ]
        }
    return result if result else {"error": f"Peak {peak_id} not found."}


def check_mz_in_peak(peak_id: int, target_mz: float, tolerance_da: float = 0.05) -> dict:
    """Checks all raw MS1 scans across a specific peak's retention time window for a target m/z."""
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found in memory."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])

    matches = []
    for spec in od:
        if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
            mzs, ints = spec.get_peaks()
            mask = (mzs >= target_mz - tolerance_da) & (mzs <= target_mz + tolerance_da)
            if np.any(mask):
                for m, inten in zip(mzs[mask], ints[mask]):
                    matches.append({
                        "rt": round(float(spec.getRT()), 2),
                        "observed_mz": round(float(m), 4),
                        "error_da": round(float(m - target_mz), 4),
                        "ppm_error": round(float((m - target_mz) / target_mz * 1e6), 1),
                        "intensity": round(float(inten), 1)
                    })

    matches = sorted(matches, key=lambda x: x["intensity"], reverse=True)
    return {
        "peak_id": peak_id,
        "rt_window": f"{rt_start:.1f}s - {rt_end:.1f}s",
        "target_mz_searched": target_mz,
        "found_in_raw_scans": len(matches) > 0,
        "max_observed_hit": matches[0] if matches else None
    }


def generate_in_silico_fragments(parent_smiles: str, check_peak_id: int = 0) -> dict:
    """
    Generates RDKit-sanitized ESI+ fragments for a parent molecule, automatically pre-renders
    the 2D SVG structure image for the parent AND every fragment (returning an `image_token`
    like `[[MOL_IMG_1]]` for each), and checks if each fragment m/z is present in `check_peak_id`.
    """
    mol, clean_parent, err = safe_mol_from_smiles(parent_smiles)
    if not mol:
        return {"error": err}

    parent_em = Descriptors.ExactMolWt(mol)
    parent_formula = rdMolDescriptors.CalcMolFormula(mol)
    parent_mh = round(parent_em + 1.007276, 4)
    parent_img_token = _register_mol_svg(mol, label="Parent Molecule")

    fragments_dict = {}

    def _add_frag(frag_smi, pathway_label, is_intrinsic_cation=False):
        f_mol, f_clean, _ = safe_mol_from_smiles(frag_smi)
        if not f_mol:
            return
        em = Descriptors.ExactMolWt(f_mol)
        if em < 40.0 or em >= parent_em - 1.0:
            return
        form = rdMolDescriptors.CalcMolFormula(f_mol)
        f_charge = Chem.GetFormalCharge(f_mol)
        if is_intrinsic_cation or f_charge > 0:
            mz_val = round(em - 0.00054858, 4)
            ion_type = "[F]+ (intrinsic cation)"
        else:
            mz_val = round(em + 1.007276, 4)
            ion_type = "[F+H]+"

        key = (form, round(mz_val, 3))
        if key not in fragments_dict:
            entry = {
                "fragment_name": pathway_label,
                "smiles": f_clean,
                "formula": form,
                "neutral_exact_mass": round(em, 4),
                "ion_type": ion_type,
                "expected_esi_pos_mz": mz_val,
                "neutral_loss_from_parent_Da": round(parent_mh - mz_val, 4),
                " _temp_mol": f_mol
            }
            if check_peak_id > 0:
                hit_info = check_mz_in_peak(check_peak_id, mz_val, tolerance_da=0.05)
                entry["present_in_peak"] = hit_info.get("found_in_raw_scans", False)
                entry["peak_observation"] = hit_info.get("max_observed_hit", None)
            fragments_dict[key] = entry

    # 1. Canonical nucleoside / ribose oxocarbenium pathways
    ribose_smarts = Chem.MolFromSmarts("[C@@H,C@H,CH]1O[C@H,C@@H,CH](CO)[C@@H,C@H,CH](O)[C@H,C@@H,CH]1")
    if mol.HasSubstructMatch(ribose_smarts):
        _add_frag("OC[C@H]1O[C+]=[C@H][C@@H]1O", "Ribose oxocarbenium fragment [C5H7O3]+", is_intrinsic_cation=True)

    # 2. Single non-ring bond cleavages
    cleavable_bonds = [b.GetIdx() for b in mol.GetBonds() if not b.IsInRing() and b.GetBondType() == Chem.BondType.SINGLE]
    for b_idx in cleavable_bonds:
        try:
            frag_mol = Chem.FragmentOnBonds(mol, [b_idx], addDummies=False)
            for f in Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False):
                _add_frag(Chem.MolToSmiles(f), "Single-bond cleavage fragment ([F+H]+)")
        except Exception:
            pass

    # 3. Purine/Adenosine cross-ring and base-loss pathways
    if "ncnc2c1ncn2" in clean_parent or "ncnc2c1nc[nH]2" in clean_parent:
        _add_frag("Nc1ncnc2c1nc[nH]2", "Protonated Adenine [B+H]+ (glycosidic cleavage, -Ribose)")
        _add_frag("c1ncnc2c1nc[nH]2", "Purine core after NH3 loss ([B+H-NH3]+)")
        _add_frag("Nc1ncnc2c1ncn2C=C", "0,2X Cross-ring cleavage (Adenine + vinyl remnant)")
        _add_frag("Nc1ncnc2c1ncn2CC=O", "0,2X Cross-ring ribose cleavage (Adenine + C2H3O)")
        _add_frag("Nc1ncnc2c1ncn2[C@@H](O)C=O", "0,3X Cross-ring ribose cleavage (Adenine + C2H3O2)")

    # Sort fragments and pre-register their 2D SVG images in order so tokens always exist!
    sorted_frags = sorted(fragments_dict.values(), key=lambda x: x["expected_esi_pos_mz"], reverse=True)
    for f_entry in sorted_frags:
        f_mol = f_entry.pop(" _temp_mol")
        f_entry["image_token"] = _register_mol_svg(f_mol, label=f_entry["fragment_name"])

    return {
        "parent_smiles": clean_parent,
        "parent_formula": parent_formula,
        "parent_exact_mass": round(parent_em, 4),
        "parent_[M+H]+_mz": parent_mh,
        "parent_image_token": parent_img_token,
        "generated_fragments": sorted_frags,
        "rendering_instruction": (
            "Each molecule above already has its 2D image pre-rendered! "
            "Simply include the exact `parent_image_token` or `image_token` (e.g. [[MOL_IMG_1]]) "
            "on its own line in your response where you want that structure displayed."
        )
    }


def analyze_structure_and_mass(smiles_or_formula: str, target_mz: float = 0.0, label: str = "") -> dict:
    """
    Calculates exact monoisotopic mass, formula, and ESI m/z values ([M]+, [M+H]+, [M+Na]+),
    and automatically pre-renders the 2D SVG structure image (returning `image_token`).
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles_or_formula)
    if mol is None:
        try:
            em = rdMolDescriptors.CalcExactMolWt(smiles_or_formula.strip())
            return {
                "input_interpreted_as": "Molecular Formula (no 2D structure)",
                "formula": smiles_or_formula.strip(),
                "exact_monoisotopic_mass": round(em, 5),
                "protonated_[M+H]+": round(em + 1.007276, 4)
            }
        except Exception:
            return {"error": err}

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    formal_charge = Chem.GetFormalCharge(mol)
    mz_mh_plus = exact_mass + 1.007276 if formal_charge == 0 else exact_mass - (formal_charge * 0.00054858)

    img_token = _register_mol_svg(mol, label=label or formula)

    res = {
        "canonical_smiles": clean_smi,
        "formula": formula,
        "exact_monoisotopic_mass": round(exact_mass, 5),
        "protonated_[M+H]+": round(mz_mh_plus, 4),
        "image_token": img_token,
        "instruction": f"Place {img_token} on its own line in your response to display the 2D structure."
    }
    if target_mz > 0:
        diff_da = mz_mh_plus - target_mz
        res["target_mz_comparison"] = {
            "target_mz": target_mz,
            "error_Da": round(diff_da, 4),
            "ppm_error": round((diff_da / target_mz) * 1e6, 1)
        }
    return res


def generate_2d_structure_svg(smiles: str, label: str = "") -> dict:
    """Explicit tool to generate a 2D chemical structure SVG and return its `image_token` (e.g. [[MOL_IMG_1]])."""
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if not mol:
        return {"error": err}
    token = _register_mol_svg(mol, label=label)
    return {
        "status": "success",
        "canonical_smiles": clean_smi,
        "image_token": token,
        "instruction": f"Insert the exact token {token} on its own line in your response to display this structure."
    }


# --- BIND TOOLS & STATEFUL CHAT UI ---

client = genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))

rdkit_tools = [
    get_user_candidate_molecules,
    get_detected_peaks_summary,
    get_peak_details,
    check_mz_in_peak,
    generate_in_silico_fragments,
    analyze_structure_and_mass,
    generate_2d_structure_svg,
]

def build_system_instruction() -> str:
    candidates_md = get_user_candidate_molecules()
    peaks_md = get_detected_peaks_summary()
    return f"""You are an expert analytical chemist and mass spectrometrist assistant integrated into an HPLC-ESI-MS / GC-MS notebook.

### Live Experimental Context:
**1. User's Candidate Molecules Table:**
{candidates_md}

**2. Deconvoluted HPLC-MS Peaks Summary (`detected_df`):**
{peaks_md}

### Mandatory Operating Rules:
1. **Use `generate_in_silico_fragments` First for Fragment Questions:**
   - Whenever the user asks for potential ESI fragments of a molecule (e.g., adenosine) and/or asks if they appear in a specific peak (e.g., Peak 2), call `generate_in_silico_fragments(parent_smiles, check_peak_id=2)`.
   - `generate_in_silico_fragments` and `analyze_structure_and_mass` AUTOMATICALLY render the 2D chemical structure SVG for every molecule and return an `image_token` (such as `[[MOL_IMG_1]]`, `[[MOL_IMG_2]]`, etc.).
2. **How to Display 2D Structure Images:**
   - Copy the exact `parent_image_token` or `image_token` (e.g., `[[MOL_IMG_1]]`) returned by the tool and place it on its own line right below each molecule or fragment description.
   - Never invent tokens that were not returned by a tool call. If you want to draw a new SMILES that wasn't in `generate_in_silico_fragments`, call `generate_2d_structure_svg(smiles, label)` or `analyze_structure_and_mass(smiles, label=...)` and use the returned `image_token`.
"""

# Static list of current stable tool-calling models (ordered newest to oldest)
curated_stable_models = [
    'gemini-3.8-flash',
    'gemini-3.6-flash',
    'gemini-3.5-flash',
    'gemini-3.5-flash-lite',
    'gemini-3.1-flash-lite',
    'gemini-2.5-pro',
    'gemini-2.5-flash',
    'gemini-2.5-flash-lite',
]

try:
    # Dynamically fetch any additional stable text/chat models available to your API key
    api_models = [
        m.name.replace('models/', '')
        for m in client.models.list()
        if 'generateContent' in (getattr(m, 'supported_actions', None) or [])
        and m.name.startswith('models/gemini-')
        and not any(x in m.name for x in ['preview', 'exp', 'image', 'tts', 'live', 'transcribe', 'embedding', '1.5', '2.0'])
    ]
    for m_id in sorted(api_models, reverse=True):
        if m_id not in curated_stable_models:
            curated_stable_models.append(m_id)
except Exception:
    pass

model_dropdown = widgets.Dropdown(
    options=curated_stable_models,
    value='gemini-3.8-flash',
    description='Model:',
    disabled=False,
    layout=widgets.Layout(width='320px')
)

reset_button = widgets.Button(
    description='Reset Chat / Refresh Data',
    button_style='warning',
    icon='refresh',
    layout=widgets.Layout(width='200px')
)

chat_input = widgets.Textarea(
    value='',
    placeholder='Ask about your HPLC-MS data or fragments (e.g., "Provide structures and masses of potential adenosine ESI fragments and check if any are present in peak 2")...',
    description='Input:',
    layout=widgets.Layout(width='85%', height='90px')
)

submit_button = widgets.Button(
    description='Send',
    button_style='primary',
    icon='paper-plane'
)

output_area = widgets.Output(
    layout=widgets.Layout(border='1px solid #e5e7eb', padding='12px', height='650px', overflow_y='auto')
)

chat_session = None
active_model_name = None
chat_history_ui = []

def init_chat_session(force_reset=False):
    global chat_session, active_model_name, chat_history_ui
    if force_reset or chat_session is None or active_model_name != model_dropdown.value:
        active_model_name = model_dropdown.value
        chat_session = client.chats.create(
            model=active_model_name,
            config=types.GenerateContentConfig(
                system_instruction=build_system_instruction(),
                tools=rdkit_tools,
                temperature=0.2,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=False,
                    maximum_remote_calls=25
                )
            )
        )
        if force_reset:
            chat_history_ui.clear()
            _rendered_svg_registry.clear()
            with output_area:
                clear_output()
                print("🔄 Chat session reset and synced with latest HPLC-MS DataFrame.")

def extract_response_text(response) -> str:
    text_chunks = []
    if response.candidates:
        for part in response.candidates[0].content.parts:
            if getattr(part, 'text', None):
                text_chunks.append(part.text)
    return "".join(text_chunks) if text_chunks else "*(No text returned.)*"

def render_message_with_svgs(markdown_text: str):
    """
    Splits the response on any [[MOL_IMG_X]] or {{SVG_X}} placeholder tokens
    and renders the stored RDKit 2D image cards via IPython.display.HTML.
    """
    pattern = r'(\[\[MOL_IMG_\d+\]\]|\{\{SVG_\d+\}\})'
    parts = re.split(pattern, markdown_text)
    for part in parts:
        part_stripped = part.strip()
        if part_stripped in _rendered_svg_registry:
            display(HTML(_rendered_svg_registry[part_stripped]))
        elif part_stripped:
            display(Markdown(part))

def on_submit_clicked(b):
    user_text = chat_input.value.strip()
    if not user_text:
        return

    chat_input.disabled = True
    submit_button.disabled = True

    init_chat_session()

    with output_area:
        print(f"👤 You: {user_text}\n")
        print(f"🤖 {model_dropdown.value} is analyzing with RDKit & MS tools...\n")

    try:
        response = chat_session.send_message(user_text)
        reply_text = extract_response_text(response)

        chat_history_ui.append(("user", user_text))
        chat_history_ui.append(("model", f"**🤖 Gemini [{model_dropdown.value}]:**\n\n{reply_text}"))

        with output_area:
            clear_output(wait=True)
            for role, msg in chat_history_ui:
                if role == "user":
                    display(Markdown(f"**👤 You:** {msg}"))
                else:
                    render_message_with_svgs(msg)
                display(HTML("<hr style='border: 0; border-top: 1px solid #374151; margin: 14px 0;'>"))

    except Exception as e:
        with output_area:
            print(f"❌ Error: {e}")

    finally:
        chat_input.value = ''
        chat_input.disabled = False
        submit_button.disabled = False

def on_reset_clicked(b):
    init_chat_session(force_reset=True)

submit_button.on_click(on_submit_clicked)
reset_button.on_click(on_reset_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>💬 Gemini + RDKit HPLC-MS Assistant</h3>"),
    widgets.HBox([model_dropdown, reset_button]),
    chat_input,
    submit_button,
    output_area
])
display(ui)

---
### Step 4: Multimodal Vision QC — Let Gemini *See* the Chromatogram

Until now, Gemini has only reasoned over numeric data (`detected_df`, dominant m/z lists). The cell below instead feeds the **actual rendered plot images** — the raw HPLC trace from Cell 4 and the deconvoluted, annotated chromatogram with embedded 2D structure cards from Cell 7 — directly into Gemini's multimodal vision context. This lets the model visually spot peak shapes and areas, baseline distortions, peak tailing/fronting, co-elution anomalies, and whether the drawn chemical structures look consistent with their assigned peaks — things a numeric table alone can miss.

In [ ]:
#@title Step 4: Multimodal Vision QC — Feed Chromatogram & Structure Images to Gemini
import io
from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import display, Markdown


def _figure_to_png_bytes(fig_or_bytes, dpi: int = 160) -> bytes:
    """Rasterizes a chromatogram image to PNG bytes. Accepts either a matplotlib `Figure`
    (rasterized via `savefig`, even if already auto-closed by the inline backend — the
    Figure object itself remains valid and renderable) or raw PNG bytes already exported
    from Cell 7's interactive Plotly chromatogram via `fig.to_image()`/kaleido, so Gemini
    sees the exact same zoomable chromatogram the user inspected."""
    if isinstance(fig_or_bytes, (bytes, bytearray)):
        return bytes(fig_or_bytes)
    buf = io.BytesIO()
    fig_or_bytes.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
    buf.seek(0)
    return buf.getvalue()


def collect_vision_qc_images() -> dict:
    """
    Gathers every chromatogram image stashed by Cells 4 & 7 in `_notebook_figure_registry`.
    Cell 4 caches a matplotlib Figure (auto-closed by the inline backend, but still
    rasterizable). Cell 7's chromatogram is an interactive Plotly figure; rather than
    re-rendering it statically, Cell 7 exports the *exact* figure the user is
    zooming/panning through to PNG bytes (via kaleido) and caches those bytes here, so
    this QC step sees precisely what the user sees.
    """
    registry = globals().get('_notebook_figure_registry', {})
    images = {}
    for name, fig in registry.items():
        try:
            images[name] = _figure_to_png_bytes(fig)
        except Exception as e:
            print(f"⚠️ Could not rasterize '{name}': {e}")
    return images


_vision_client = globals().get('client') or genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))
_vision_model_name = globals()['model_dropdown'].value if 'model_dropdown' in globals() else 'gemini-3.8-flash'

VISION_QC_PROMPT = """You are an expert analytical chemist visually inspecting these HPLC/MS chromatogram images (not just numeric data). Examine every image provided and report, peak-by-peak where possible:

1. **Peaks & areas** — how many distinct peaks are visible, their approximate retention times, and relative areas.
2. **Baseline distortions** — drift, noise, humps, or a non-zero baseline between peaks.
3. **Peak tailing / fronting** — any visible asymmetry versus an ideal Gaussian peak shape.
4. **Co-elution anomalies** — shoulders, unresolved doublets, or overlapping peaks that may hide a second compound.
5. **Chemical structures** — for any 2D molecular structure drawings embedded on the chromatogram, describe the structure and flag if it looks inconsistent with the peak it is attached to (e.g. wrong ring system, mismatched size).

Present your findings as a concise markdown report with one section per image."""


def run_multimodal_chromatogram_vision_qc(user_question: str = None) -> str:
    """Sends the cached chromatogram/structure images into Gemini's vision context and displays its QC report."""
    images = collect_vision_qc_images()
    if not images:
        msg = "⚠️ No chromatogram figures found in `_notebook_figure_registry`. Run Cells 4 and 7 first."
        display(Markdown(msg))
        return msg

    parts = [types.Part.from_text(text=user_question or VISION_QC_PROMPT)]
    for name, png_bytes in images.items():
        parts.append(types.Part.from_text(text=f"--- Image: {name} ---"))
        parts.append(types.Part.from_bytes(data=png_bytes, mime_type='image/png'))

    response = _vision_client.models.generate_content(
        model=_vision_model_name,
        contents=[types.Content(role='user', parts=parts)],
    )

    report = getattr(response, 'text', None) or "*(No text returned.)*"
    display(Markdown(f"### 🔬 Gemini Multimodal Vision QC Report ({len(images)} image(s) analyzed)\n\n{report}"))
    return report


# Run once automatically with the default QC prompt.
# Re-run any time with a custom, targeted question, e.g.:
# run_multimodal_chromatogram_vision_qc("Does the structure card on Peak 2 look right for adenosine?")
_ = run_multimodal_chromatogram_vision_qc()